# NRPy 2: A 10-Minute Overview

## Author: Zach Etienne

To run the whole notebook, click the `>>` toolbar button and choose
**Restart Kernel and Run All Cells…**.

This notebook gives a compact tour of NRPy 2: symbolic tensor construction with
Python and SymPy, optimized C code generation, C-function registration, and
finite-difference-aware kernel output. The running example constructs the 18
independent spatial Christoffel symbols from a symmetric three-metric.

Navigation: [Index](../index.ipynb) |
Previous: [Getting Started with NRPy](../0-getting_started/install_and_run.ipynb) |
Next: [Parameters](params.ipynb)


# Table of Contents

This notebook is organized as follows:

1. [Part 1](#Part-1:-Why-NRPy?): Why NRPy?
1. [Part 2](#Part-2:-Construct-spatial-Christoffels): Construct spatial
   Christoffels from the three-metric.
1. [Part 3](#Part-3:-Generate-optimized-C): Generate optimized C for the
   Christoffel expressions.
1. [Part 4](#Part-4:-Register-a-C-function): Register a C function that computes
   Christoffel symbols.
1. [Part 5](#Part-5:-Generate-finite-difference-aware-C): Generate
   finite-difference-aware C.
1. [Part 6](#Part-6:-What-next?): What next?


# Part 1: Why NRPy?
### \[Back to [top](#Table-of-Contents)\]

Numerical relativity often uses standard numerical methods, but the equations
are large, tensorial, and difficult to implement by hand without errors.

[Einstein notation](https://en.wikipedia.org/wiki/Einstein_notation) makes this
structure manageable on paper by suppressing explicit tensor sums.

NRPy 2 treats tensor equations written in Einstein notation as executable
symbolic specifications. In Python,

* rank-0 tensors (scalars) are *variables*
* rank-1 tensors are *lists*
* rank-2 tensors are *lists of lists*
* ... and so forth.

Implied tensor summations become explicit Python loops.

NRPy combines this representation with [SymPy](https://www.sympy.org/) and core
code-generation modules that convert symbolic expressions into optimized C
kernels. Its finite-difference support can also insert stencil reads directly
into generated code.

The NRPy 2 tutorial proceeds from setup through core modules, finite
differences, wave-equation examples, curvilinear coordinates, infrastructure
targets, and numerical-relativity workflows. This overview focuses on the
symbolic-to-C pipeline used throughout those notebooks.


# Part 2: Construct spatial Christoffels
### \[Back to [top](#Table-of-Contents)\]

**Problem statement**: Given a three-metric $\gamma_{ij}$, construct the 18
independent Christoffel symbols $\Gamma^i_{jk}$. For now, assume that
$\gamma_{ij}$ and its first derivatives are already available as symbols.

NRPy represents tensors and indexed expressions as nested Python lists, so for
example

* $\gamma_{ij}=$ `gammaDD[i][j]`
* $\gamma_{ij,k}=$ `gammaDD_dD[i][j][k]`

Christoffel symbols are defined by

\begin{align}
\Gamma_{ij}^k &= \frac{1}{2} \gamma^{kl}
    \left(\gamma_{jl,i} + \gamma_{il,j} - \gamma_{ij,l}\right).
\end{align}

First define $\gamma_{ij}$ and its inverse with NRPy's indexed-expression
helpers.


In [1]:
import sympy as sp
import nrpy.indexedexp as ixp

# gammaDD is a rank-2 indexed expression symmetric in indices 0 and 1.
gammaDD = ixp.declarerank2("gammaDD", symmetry="sym01", dimension=3)

# gammaUU is the inverse of gammaDD; gammaDET is its determinant.
gammaUU, gammaDET = ixp.symm_matrix_inverter3x3(gammaDD)

The generated objects encode the requested symmetry. The inverse metric is also
symmetric, and $\gamma_{ij}\gamma^{ji}=3$ for a three-dimensional metric.

**Exercise**: Replace the final trace check below with explicit loops over all
dimensions, then simplify the result.


In [2]:
print("Check that gammaDD[0][1] = gammaDD[1][0]:")
print("gammaDD[0][1] = ", gammaDD[0][1])
print("gammaDD[1][0] = ", gammaDD[1][0])
assert gammaDD[0][1] == gammaDD[1][0]

print("\nOutput gammaUU[1][0] = ", gammaUU[1][0])
print("\nCheck that gammaUU[1][0] - gammaUU[0][1] = 0:")
difference = sp.simplify(gammaUU[1][0] - gammaUU[0][1])
print("gammaUU[1][0] - gammaUU[0][1] = ", difference)
assert difference == 0

trace = sp.simplify(sum(gammaDD[i][j] * gammaUU[j][i] for i in range(3) for j in range(3)))
print("\nTrace gammaDD[i][j] * gammaUU[j][i] = ", trace)
assert trace == 3

Check that gammaDD[0][1] = gammaDD[1][0]:
gammaDD[0][1] =  gammaDD01
gammaDD[1][0] =  gammaDD01

Output gammaUU[1][0] =  (-gammaDD01*gammaDD22 + gammaDD02*gammaDD12)/(gammaDD00*gammaDD11*gammaDD22 - gammaDD00*gammaDD12**2 - gammaDD01**2*gammaDD22 + 2*gammaDD01*gammaDD02*gammaDD12 - gammaDD02**2*gammaDD11)

Check that gammaUU[1][0] - gammaUU[0][1] = 0:
gammaUU[1][0] - gammaUU[0][1] =  0

Trace gammaDD[i][j] * gammaUU[j][i] =  3


Define Christoffel symbols in terms of the inverse metric and metric first derivatives:
$$
\Gamma_{ij}^k = \frac{1}{2} \gamma^{kl}\left(\gamma_{jl,i} + \gamma_{il,j} - \gamma_{ij,l}\right)
$$

In [3]:
# First define symbolic expressions for metric derivatives.
gammaDD_dD = ixp.declarerank3("gammaDD_dD", symmetry="sym01", dimension=3)

# Initialize GammaUDD (spatial Christoffel symbols) to zero.
GammaUDD = ixp.zerorank3(dimension=3)
for i in range(3):
    for j in range(3):
        for k in range(3):
            for l in range(3):
                GammaUDD[k][i][j] += sp.Rational(1, 2) * gammaUU[k][l] * (
                    gammaDD_dD[j][l][i]
                    + gammaDD_dD[i][l][j]
                    - gammaDD_dD[i][j][l]
                )

Now confirm the lower-index symmetry $\Gamma^k_{ij}=\Gamma^k_{ji}$.

In [4]:
assert all(
    sp.simplify(GammaUDD[k][i][j] - GammaUDD[k][j][i]) == 0
    for i in range(3)
    for j in range(3)
    for k in range(3)
)
print("All GammaUDD[k][i][j] - GammaUDD[k][j][i] symmetry checks vanish.")

All GammaUDD[k][i][j] - GammaUDD[k][j][i] symmetry checks vanish.


# Part 3: Generate optimized C
### \[Back to [top](#Table-of-Contents)\]

At the core of NRPy is the ability to convert SymPy expressions to optimized C
code.

**Problem statement**: Output all 18 unique Christoffel symbols with three
optimization modes supported by NRPy's `c_codegen()`.

First store all 18 unique Christoffel symbols and their desired C variable names
in Python lists.


In [5]:
symbols_list = []
varname_list = []
for i in range(3):
    for j in range(3):
        for k in range(j, 3):
            symbols_list.append(GammaUDD[i][j][k])
            varname_list.append(f"ChristoffelUDD{i}{j}{k}")

assert len(symbols_list) == 18
assert len(varname_list) == 18
print(f"Prepared {len(symbols_list)} independent Christoffel expressions.")

Prepared 18 independent Christoffel expressions.


Next pass these lists to NRPy's C code-generation function `c_codegen()`.

**Optimization Level 0**: compute each Christoffel symbol independently. This
exposes repeated subexpressions.


In [6]:
import nrpy.c_codegen as ccg


def print_code_excerpt(label, code, head_lines=0, markers=()):
    """Print selected lines from generated C code."""
    lines = code.splitlines()
    selected = list(range(min(head_lines, len(lines))))
    for marker in markers:
        for index, line in enumerate(lines):
            if marker in line:
                if index not in selected:
                    selected.append(index)
                break

    print(f"{label} ({len(selected)} of {len(lines)} selected lines shown):")
    previous = None
    for index in sorted(selected):
        if previous is not None and index > previous + 1:
            print("...")
        print(lines[index])
        previous = index


code_no_cse = ccg.c_codegen(
    symbols_list,
    varname_list,
    enable_cse=False,
    verbose=False,
    enable_clang_format=False,
    include_braces=False,
)
print_code_excerpt("No CSE", code_no_cse, markers=("ChristoffelUDD000",))


No CSE (1 of 18 selected lines shown):
ChristoffelUDD000 = (1.0/2.0)*gammaDD_dD000*(gammaDD11*gammaDD22 - ((gammaDD12)*(gammaDD12)))/(gammaDD00*gammaDD11*gammaDD22 - gammaDD00*((gammaDD12)*(gammaDD12)) - ((gammaDD01)*(gammaDD01))*gammaDD22 + 2*gammaDD01*gammaDD02*gammaDD12 - ((gammaDD02)*(gammaDD02))*gammaDD11) + (1.0/2.0)*(-gammaDD_dD001 + 2*gammaDD_dD010)*(-gammaDD01*gammaDD22 + gammaDD02*gammaDD12)/(gammaDD00*gammaDD11*gammaDD22 - gammaDD00*((gammaDD12)*(gammaDD12)) - ((gammaDD01)*(gammaDD01))*gammaDD22 + 2*gammaDD01*gammaDD02*gammaDD12 - ((gammaDD02)*(gammaDD02))*gammaDD11) + (1.0/2.0)*(-gammaDD_dD002 + 2*gammaDD_dD020)*(gammaDD01*gammaDD12 - gammaDD02*gammaDD11)/(gammaDD00*gammaDD11*gammaDD22 - gammaDD00*((gammaDD12)*(gammaDD12)) - ((gammaDD01)*(gammaDD01))*gammaDD22 + 2*gammaDD01*gammaDD02*gammaDD12 - ((gammaDD02)*(gammaDD02))*gammaDD11);


The unoptimized code recomputes many factors. Common subexpression elimination
introduces temporaries for repeated symbolic subexpressions.

**Optimization Level 1**: use
[common subexpression elimination](https://en.wikipedia.org/wiki/Common_subexpression_elimination).


In [7]:
code_cse = ccg.c_codegen(
    symbols_list,
    varname_list,
    enable_cse=True,
    verbose=False,
    enable_clang_format=False,
    include_braces=False,
)
assert "ChristoffelUDD" in code_cse
print_code_excerpt("CSE", code_cse, head_lines=6, markers=("ChristoffelUDD000",))

CSE (7 of 34 selected lines shown):
const REAL tmp5 = -gammaDD_dD001 + 2*gammaDD_dD010;
const REAL tmp7 = -gammaDD_dD002 + 2*gammaDD_dD020;
const REAL tmp9 = -gammaDD_dD012 + gammaDD_dD021 + gammaDD_dD120;
const REAL tmp10 = gammaDD_dD012 - gammaDD_dD021 + gammaDD_dD120;
const REAL tmp11 = -gammaDD_dD112 + 2*gammaDD_dD121;
const REAL tmp12 = 2*gammaDD_dD011 - gammaDD_dD110;
...
ChristoffelUDD000 = gammaDD_dD000*tmp4 + tmp5*tmp6 + tmp7*tmp8;


**Optimization Level 2**: use CSE and
[single instruction, multiple data](https://en.wikipedia.org/wiki/Single_instruction,_multiple_data)
helper macros. This form is intended for kernels evaluated over numerical grid
data.


In [8]:
code_simd = ccg.c_codegen(
    symbols_list,
    varname_list,
    enable_cse=True,
    enable_simd=True,
    verbose=False,
    enable_clang_format=False,
    include_braces=False,
)
print_code_excerpt(
    "CSE plus SIMD", code_simd, head_lines=14, markers=("ChristoffelUDD000",)
)

CSE plus SIMD (15 of 48 selected lines shown):
static const double dbl_Integer_1 = 1.0;
MAYBE_UNUSED const REAL_SIMD_ARRAY _Integer_1 = ConstSIMD(dbl_Integer_1);

static const double dbl_Integer_2 = 2.0;
 const REAL_SIMD_ARRAY _Integer_2 = ConstSIMD(dbl_Integer_2);

static const double dbl_NegativeOne_ = -1.0;
MAYBE_UNUSED const REAL_SIMD_ARRAY _NegativeOne_ = ConstSIMD(dbl_NegativeOne_);

static const double dbl_Rational_1_2 = 1.0/2.0;
 const REAL_SIMD_ARRAY _Rational_1_2 = ConstSIMD(dbl_Rational_1_2);

const REAL_SIMD_ARRAY tmp1 = MulSIMD(_NegativeOne_, MulSIMD(gammaDD12, gammaDD12));
const REAL_SIMD_ARRAY tmp3 = MulSIMD(_NegativeOne_, MulSIMD(gammaDD01, gammaDD01));
...
ChristoffelUDD000 = FusedMulAddSIMD(tmp10, tmp9, FusedMulAddSIMD(tmp7, tmp8, MulSIMD(gammaDD_dD000, tmp6)));


# Part 4: Register a C function
### \[Back to [top](#Table-of-Contents)\]

Generated assignments become more useful when they are placed inside a complete C
function. A C function has a fixed outer structure:

```c
/*
 * Description of what the function computes.
 */
return_type function_name(parameter_list) {
    local_declarations;
    generated_assignments;
}
```

NRPy's `register_CFunction()` mirrors this structure. The function description
maps to `desc`, the return type maps to `cfunc_type`, the C name maps to `name`,
the input/output list maps to `params`, and the generated assignment block maps
to `body`. The `includes` list records headers needed when the generated project
writes this function to source files.

The next cell follows the BHaH-style registration pattern: define the C-function
pieces as separate Python variables, generate the Christoffel assignment body,
then pass those pieces to `register_CFunction()`.


In [9]:
import nrpy.c_function as cfc

# Step 1: Generate the body that writes the independent Christoffel components.
christoffel_output_varnames = [
    f"GammaUDD[{component}]"
    for component in range(len(symbols_list))
]
body = ccg.c_codegen(
    symbols_list,
    christoffel_output_varnames,
    enable_cse=True,
    verbose=False,
    include_braces=False,
)

# Step 2: Define the C-function pieces, matching the BHaH registration pattern.
includes = ["BHaH_defines.h"]
desc = r"""Compute the 18 independent spatial Christoffel symbols Gamma^i_{jk}."""
cfunc_type = "void"
name = "compute_Christoffel_symbols"
arg_dict = {
    "gammaDD00": "const REAL",
    "gammaDD01": "const REAL",
    "gammaDD02": "const REAL",
    "gammaDD11": "const REAL",
    "gammaDD12": "const REAL",
    "gammaDD22": "const REAL",
}
for i in range(3):
    for j in range(i, 3):
        for k in range(3):
            arg_dict[f"gammaDD_dD{i}{j}{k}"] = "const REAL"
arg_dict["GammaUDD"] = "REAL *restrict"
params = ", ".join([f"{value} {key}" for key, value in arg_dict.items()])

# Step 3: Register the function in NRPy's C-function dictionary.
cfc.CFunction_dict.pop(name, None)
cfc.register_CFunction(
    includes=includes,
    desc=desc,
    cfunc_type=cfunc_type,
    name=name,
    params=params,
    include_CodeParameters_h=False,
    body=body,
)
registered_function = cfc.CFunction_dict[name]

# Step 4: Validate and inspect the registered function.
if registered_function.name != name:
    raise RuntimeError(f"Unexpected function name: {registered_function.name}")
if registered_function.params != params:
    raise RuntimeError(f"Unexpected function parameters: {registered_function.params}")
if registered_function.body != body:
    raise RuntimeError("Stored C-function body does not match generated body.")
if len(christoffel_output_varnames) != 18:
    raise RuntimeError("Expected 18 independent Christoffel outputs.")
for expected_piece in ["GammaUDD[0]", "END FUNCTION: compute_Christoffel_symbols"]:
    if expected_piece not in registered_function.full_function:
        raise RuntimeError(f"Missing {expected_piece} in registered function.")

print("registered function declaration:")
print(registered_function.function_prototype)
print_code_excerpt(
    "registered Christoffel body",
    registered_function.body,
    head_lines=8,
    markers=("GammaUDD[0]",),
)
print("validated registered function:", registered_function.name)


registered function declaration:
void compute_Christoffel_symbols(const REAL gammaDD00, const REAL gammaDD01, const REAL gammaDD02, const REAL gammaDD11, const REAL gammaDD12, const REAL gammaDD22, const REAL gammaDD_dD000, const REAL gammaDD_dD001, const REAL gammaDD_dD002, const REAL gammaDD_dD010, const REAL gammaDD_dD011, const REAL gammaDD_dD012, const REAL gammaDD_dD020, const REAL gammaDD_dD021, const REAL gammaDD_dD022, const REAL gammaDD_dD110, const REAL gammaDD_dD111, const REAL gammaDD_dD112, const REAL gammaDD_dD120, const REAL gammaDD_dD121, const REAL gammaDD_dD122, const REAL gammaDD_dD220, const REAL gammaDD_dD221, const REAL gammaDD_dD222, REAL *restrict GammaUDD);
registered Christoffel body (9 of 34 selected lines shown):
const REAL tmp5 = -gammaDD_dD001 + 2*gammaDD_dD010;
const REAL tmp7 = -gammaDD_dD002 + 2*gammaDD_dD020;
const REAL tmp9 = -gammaDD_dD012 + gammaDD_dD021 + gammaDD_dD120;
const REAL tmp10 = gammaDD_dD012 - gammaDD_dD021 + gammaDD_dD120;
const REAL t

# Part 5: Generate finite-difference-aware C
### \[Back to [top](#Table-of-Contents)\]

The previous C code assumed that $\gamma_{ij,k}$ values had already been
computed. NRPy's finite-difference-aware code generation can instead insert
stencil expressions for derivative symbols.

Here we use a uniform-grid finite-difference example. The input metric
components are registered as grid fields, and the independent Christoffel
components are registered as output grid fields.

The parameter `fd_order=6` selects sixth-order finite-difference stencils.
`Infrastructure="BHaH"` selects the memory-access convention used in this
generated kernel.

The example keeps CSE enabled and omits SIMD so the stencil structure remains
compact enough to inspect.


In [10]:
import nrpy.grid as gri
import nrpy.params as par

gri.glb_gridfcs_dict.clear()
par.set_parval_from_str("Infrastructure", "BHaH")
par.set_parval_from_str("fd_order", 6)

gri.register_gridfunctions_for_single_rank2(
    "gammaDD", group="EVOL", dimension=3, symmetry="sym01"
)

christoffel_gf_names = list(varname_list)
gri.register_gridfunctions(
    christoffel_gf_names,
    group="AUX",
    dimension=3,
    rank=3,
    is_basename=False,
)

output_varnames = [
    gri.BHaHGridFunction.access_gf(gf_name=name, gf_array_name="aux_gfs")
    for name in christoffel_gf_names
]
fd_code = ccg.c_codegen(
    symbols_list,
    output_varnames,
    enable_cse=True,
    enable_fd_codegen=True,
    verbose=False,
    enable_clang_format=False,
    include_braces=False,
)

for marker in [
    "NRPy-Generated GF Access/FD Code",
    "IDX4",
    "aux_gfs[IDX4(CHRISTOFFELUDD000GF",
    "gammaDD_dD000",
]:
    assert marker in fd_code, marker

print_code_excerpt(
    "Finite-difference C code",
    fd_code,
    head_lines=12,
    markers=("gammaDD_dD000", "aux_gfs[IDX4(CHRISTOFFELUDD000GF"),
)

Finite-difference C code (14 of 178 selected lines shown):
/*
 * NRPy-Generated GF Access/FD Code, Step 1 of 2:
 * Read gridfunction(s) from main memory and compute FD stencils as needed.
 */
const REAL gammaDD00_i2m3 = in_gfs[IDX4(GAMMADD00GF, i0, i1, i2-3)];
const REAL gammaDD00_i2m2 = in_gfs[IDX4(GAMMADD00GF, i0, i1, i2-2)];
const REAL gammaDD00_i2m1 = in_gfs[IDX4(GAMMADD00GF, i0, i1, i2-1)];
const REAL gammaDD00_i1m3 = in_gfs[IDX4(GAMMADD00GF, i0, i1-3, i2)];
const REAL gammaDD00_i1m2 = in_gfs[IDX4(GAMMADD00GF, i0, i1-2, i2)];
const REAL gammaDD00_i1m1 = in_gfs[IDX4(GAMMADD00GF, i0, i1-1, i2)];
const REAL gammaDD00_i0m3 = in_gfs[IDX4(GAMMADD00GF, i0-3, i1, i2)];
const REAL gammaDD00_i0m2 = in_gfs[IDX4(GAMMADD00GF, i0-2, i1, i2)];
...
const REAL gammaDD_dD000 = invdxx0*(FDPart1_Rational_1_60*(-gammaDD00_i0m3 + gammaDD00_i0p3) + FDPart1_Rational_3_20*(gammaDD00_i0m2 - gammaDD00_i0p2) + FDPart1_Rational_3_4*(-gammaDD00_i0m1 + gammaDD00_i0p1));
...
aux_gfs[IDX4(CHRISTOFFELUDD000GF, i0,

# Part 6: What next?
### \[Back to [top](#Table-of-Contents)\]

Continue with the notebooks most relevant to your next task:

- [Repository index](../index.ipynb)
- [Parameters](params.ipynb)
- [Indexed expressions](indexedexp.ipynb)
- [Grid fields](grid.ipynb)
- [C code generation](c_codegen.ipynb)
- [C function registration](c_function.ipynb)
- [Finite differences](finite_difference.ipynb)
- [Wave equation with NumPy](../2-numerical_methods/wave_equation_with_numpy.ipynb)
- [Wave equation with C code generation](../3-wave_equation/wave_equation_and_c_codegen.ipynb)
